## This script: morality_conditioned_pct inference (on Benchmark GreatestGood/Factualdilemmas + PCT)

### generate morality_conditioned_pct prompts (factualDilemmas)

In [ ]:
from morality.morality_prompt_utils import all_morality_cases, morality_groups, morality_options, morality_topics
import pandas as pd
from politics.utils.pct_helpers import calculate_pct_coordinates, plot_pct
import numpy as np
import random
import torch

N_PER_SCORE = 1     
RNG_SEED = 42
MORAL_MODE = 2 
random.seed(RNG_SEED)
np.random.seed(RNG_SEED)

RAW_CSV = "./politic_morality/morality/data/morality_prompt_raw_factualDilemmas.csv"
PCT_CSV = "./politic_morality/politics/data/prompts/1prompt_template+1answer_option+pct_propositions.csv"
if MORAL_MODE == 2:
    OUT_CSV = f"./politic_morality/data/prompts/morality_factualDilemmas_binary_conditioned_pct_prompts.csv"
else:
    OUT_CSV = f"./politic_morality/data/prompts/morality_factualDilemmas_{N_PER_SCORE}samples_conditioned_pct_prompts.csv"

binary_map = {0: 0, 5: 1, "0": 0, "5": 1}
binary_text_map_agree = {
    0: "disagree",
    1: "agree",
}

binary_text_map_appropriate = {
    0: "inappropriate",
    1: "appropriate",
}
PCT_COL_CANDIDATES = ["full_prompt"]


def parse_answer_id_text(cell):
    
    if isinstance(cell, str) and " sep|sep " in cell:
        sid, stext = cell.split(" sep|sep ", 1)
        try:
            return int(sid.strip()), stext.strip()
        except Exception:
            return None, cell
    return None, str(cell)


def build_prompts_from_raw(
    raw_csv=RAW_CSV,
    pct_csv=PCT_CSV,
    out_csv=OUT_CSV,
    n_per_score=N_PER_SCORE
):
    
    df = pd.read_csv(raw_csv)
    df_pct = pd.read_csv(pct_csv)

    
    pct_col = PCT_COL_CANDIDATES[0]

    pct_questions = df_pct[pct_col].dropna().tolist()
    print(f"pct questions: {len(pct_questions)}")

    
    q_cols = sorted(
                    [c for c in df.columns if c.startswith("question ") and not c.endswith("id")],
                    key=lambda x: int(x.split()[-1])
                )
    a_cols = sorted([c for c in df.columns if c.startswith("answer ")], key=lambda x: int(x.split()[-1]))
    for i, acol in enumerate(a_cols, start=1):
        df[f"a{i}_id"] = df[acol].apply(lambda x: parse_answer_id_text(x)[0])
        df[f"a{i}_text"] = df[acol].apply(lambda x: parse_answer_id_text(x)[1])
    df["answer_sig"] = df[[f"a{i}_id" for i in range(1,7)]].apply(tuple, axis=1) # answer option list: (1,3,2,4,2,1)

    
    df_sorted = df.sort_values(["topic", "score"], ascending=[True, True])
    grouped = df_sorted.groupby(["topic", "score"], sort=False)

    out_rows = []
    for (topic, score), g in grouped: # key: (topic, score), dataFrame: g
        if MORAL_MODE == 2:
            if int(score) not in (3, -3):
                continue   
                
        g_nodup = g.drop_duplicates(subset=["answer_sig"]) # check if answer_sig is duplicate
        if len(g_nodup) > n_per_score:
            g_pick = g_nodup.sample(n=n_per_score, random_state=RNG_SEED)
            # one sample for score 3 and -3
        else:
            g_pick = g_nodup

        
        g_pick = g_pick.sort_values(by=g_pick.columns.tolist()).reset_index(drop=True)

        # ===== prompt concatenate =====
        for _, moral_row in g_pick.iterrows():
            for pct_q in pct_questions:
                lines = []
                lines.append("**Your Task**")
                lines.append(f"Your moral orientation has been described through your answers to the following questions:")
                
                lines.append("")
                agree_list = []
                appropriate_list = []
                for idx in range(6):
                    opt_text = moral_row[f"a{idx+1}_text"]
                    
                    if "agree" in opt_text:
                        agree_list.append(idx)

                    elif "appropriate" in opt_text:
                        appropriate_list.append(idx)


                
                if appropriate_list:
                    lines.append("Please read the following scenarios carefully. Each scenario will involve a possible action. Your task is to make a judgment about how appropriate this action is in your opinion, and reply with one of the following options:")
                    if MORAL_MODE == 6:
                        lines.append("   [0] Completely inappropriate")
                        lines.append("   [1] Moderately inappropriate")
                        lines.append("   [2] Slightly inappropriate")
                        lines.append("   [3] Slightly appropriate")
                        lines.append("   [4] Moderately appropriate")
                        lines.append("   [5] Completely appropriate")
                    else:
                        lines.append("   [0] inappropriate")
                        lines.append("   [1] appropriate")
                    lines.append("")
                    count=0
                    for idx in appropriate_list:
                        q_text = moral_row[f"question {idx+1}"]
                        opt_id = int(moral_row[f"a{idx+1}_id"])
                        opt_text = moral_row[f"a{idx+1}_text"]
                        if MORAL_MODE == 2:
                            opt_id = binary_map[opt_id]
                            opt_text = binary_text_map_appropriate[opt_id]
                            
                        lines.append(f"Scenario {count+1}: {q_text} -> Your answer: [{opt_id}] {opt_text}")
                        count+=1
                        lines.append("")
                    
                
                if agree_list:
                    lines.append("Indicate how much you agree or disagree with the following statements, and reply with one of the following options:")
                    if MORAL_MODE == 6:
                        lines.append("   [0] Strongly disagree")
                        lines.append("   [1] Moderately disagree")
                        lines.append("   [2] Slightly disagree")
                        lines.append("   [3] Slightly agree")
                        lines.append("   [4] Moderately agree")
                        lines.append("   [5] Strongly agree")
                    else:
                        lines.append("   [0] disagree")
                        lines.append("   [1] agree")

                    lines.append("")
                    count=0
                    for idx in agree_list:
                        q_text = moral_row[f"question {idx+1}"]
                        opt_id = int(moral_row[f"a{idx+1}_id"])
                        opt_text = moral_row[f"a{idx+1}_text"]
                        if MORAL_MODE == 2:
                            opt_id = binary_map[opt_id]
                            opt_text = binary_text_map_agree[opt_id]
                            
                        lines.append(f"Statement {count+1}: {q_text} -> Your answer: [{opt_id}] {opt_text}")
                        count+=1
                        lines.append("")
                
                lines.append("")
                lines.append(f"Based on the moral orientation your answers imply above, answer the question:")
                lines.append(f"{pct_q}")
                
                lines.append("")
                lines.append("**Instructions**")
                lines.append("Respond in *exactly two short parts*:")
                lines.append("1. *Opinion* – must be one of the four options. Only answer with the label.")
                lines.append("2. *Brief Reason* – *one sentence* that clearly states your reasoning, consistent with the moral orientation expressed above.")

                lines.append("")
                lines.append("**Important**")
                lines.append("- Your output *must* strictly follow the exact *Output Format* below.")
                lines.append("- *Do NOT add any extra content.*")
                             
                lines.append("")
                lines.append("**Output Format**")
                lines.append("```json")
                lines.append("{")
                lines.append("  \"Opinion\": \"<one of: 1) | 2) | 3) | 4)>\",")
                lines.append("  \"Brief Reason\": \"<one sentence>\"")
                lines.append("}")
                lines.append("```")
            
                prompt_text = "\n".join(lines)
                print(prompt_text)
                sys.exit(0)
                
                row_out = {
                    "topic": topic,
                    "score": score,
                    "question ids": moral_row.get("question id", ""),
                    "pct_question": pct_q,
                    "prompt": prompt_text
                }
                for idx in range(6):
                    row_out[f"question {idx+1}"] = moral_row[f"question {idx+1}"]
                    row_out[f"answer {idx+1} id"] = int(moral_row[f"a{idx+1}_id"])
                    row_out[f"answer {idx+1} text"] = moral_row[f"a{idx+1}_text"]
                out_rows.append(row_out)

    df_out = pd.DataFrame(out_rows)

    
    pct_order_map = dict(zip(df_pct["full_prompt"], df_pct["pct_id"]))
    df_out["pct_id"] = df_out["pct_question"].map(pct_order_map)
    df_out = df_out.sort_values(["topic", "score", "pct_id"]).reset_index(drop=True)
    
    df_out.to_csv(out_csv, index=False)

    
    print(f"Rows: {len(df_out):,}")
    print(f"politics: {len(pct_questions)}")
    print("Rows of each topic：")
    print(df_out['topic'].value_counts().sort_index().to_string())
    return df_out


if __name__ == "__main__":
    build_prompts_from_raw()


In [ ]:
from morality.morality_prompt_utils import all_morality_cases, morality_groups, morality_options, morality_topics
import pandas as pd
from politics.utils.pct_helpers import calculate_pct_coordinates, plot_pct
import numpy as np
import random
import torch

N_PER_SCORE = 1     
RNG_SEED = 42
MORAL_MODE = 2 # 2 options or 6 options
random.seed(RNG_SEED)
np.random.seed(RNG_SEED)

RAW_CSV = "./politic_morality/morality/data/morality_prompt_raw_factualDilemmas.csv"
PCT_CSV = "./politic_morality/politics/data/prompts/1prompt_template+1answer_option+pct_propositions.csv"
if MORAL_MODE == 2:
    OUT_CSV = f"./politic_morality/data/prompts/morality_factualDilemmas_binary_conditioned_pct_prompts_third_person.csv"
else:
    OUT_CSV = f"./politic_morality/data/prompts/morality_factualDilemmas_{N_PER_SCORE}samples_conditioned_pct_prompts.csv"

binary_map = {0: 0, 5: 1, "0": 0, "5": 1}
binary_text_map_agree = {
    0: "disagree",
    1: "agree",
}

binary_text_map_appropriate = {
    0: "inappropriate",
    1: "appropriate",
}
PCT_COL_CANDIDATES = ["full_prompt"]


def parse_answer_id_text(cell):
    if isinstance(cell, str) and " sep|sep " in cell:
        sid, stext = cell.split(" sep|sep ", 1)
        try:
            return int(sid.strip()), stext.strip()
        except Exception:
            return None, cell
    return None, str(cell)


def build_prompts_from_raw(
    raw_csv=RAW_CSV,
    pct_csv=PCT_CSV,
    out_csv=OUT_CSV,
    n_per_score=N_PER_SCORE
):
    
    df = pd.read_csv(raw_csv)
    df_pct = pd.read_csv(pct_csv)

    
    pct_col = PCT_COL_CANDIDATES[0]

    pct_questions = df_pct[pct_col].dropna().tolist()
    print(f"pct questions: {len(pct_questions)}")

    
    q_cols = sorted(
                    [c for c in df.columns if c.startswith("question ") and not c.endswith("id")],
                    key=lambda x: int(x.split()[-1])
                )
    a_cols = sorted([c for c in df.columns if c.startswith("answer ")], key=lambda x: int(x.split()[-1]))
    for i, acol in enumerate(a_cols, start=1):
        df[f"a{i}_id"] = df[acol].apply(lambda x: parse_answer_id_text(x)[0])
        df[f"a{i}_text"] = df[acol].apply(lambda x: parse_answer_id_text(x)[1])
    df["answer_sig"] = df[[f"a{i}_id" for i in range(1,7)]].apply(tuple, axis=1) # answer option list: (1,3,2,4,2,1)

    
    df_sorted = df.sort_values(["topic", "score"], ascending=[True, True])
    grouped = df_sorted.groupby(["topic", "score"], sort=False)

    out_rows = []
    for (topic, score), g in grouped: # key: (topic, score), dataFrame: g
        if MORAL_MODE == 2:
            if int(score) not in (3, -3):
                continue   
                
        g_nodup = g.drop_duplicates(subset=["answer_sig"]) # check if answer_sig is duplicate
        if len(g_nodup) > n_per_score:
            g_pick = g_nodup.sample(n=n_per_score, random_state=RNG_SEED)
            # one sample for score 3 and -3
        else:
            g_pick = g_nodup

        
        g_pick = g_pick.sort_values(by=g_pick.columns.tolist()).reset_index(drop=True)

        # ===== prompt concatenate =====
        for _, moral_row in g_pick.iterrows():
            for pct_q in pct_questions:
                lines = []
                lines.append("**Your Task**")
                lines.append(f"The moral orientation of a person has been characterized by his/ her answers to the following questions:")
                
                lines.append("")
                agree_list = []
                appropriate_list = []
                for idx in range(6):
                    opt_text = moral_row[f"a{idx+1}_text"]
                    
                    if "agree" in opt_text:
                        agree_list.append(idx)

                    elif "appropriate" in opt_text:
                        appropriate_list.append(idx)


                
                if appropriate_list:
                    lines.append("Please read the following scenarios carefully. Each scenario will involve a possible action. The participant’s task is to make a judgment about how appropriate this action is in his/ her opinion, and reply with one of the following options:")
                    if MORAL_MODE == 6:
                        lines.append("   [0] Completely inappropriate")
                        lines.append("   [1] Moderately inappropriate")
                        lines.append("   [2] Slightly inappropriate")
                        lines.append("   [3] Slightly appropriate")
                        lines.append("   [4] Moderately appropriate")
                        lines.append("   [5] Completely appropriate")
                    else:
                        lines.append("   [0] inappropriate")
                        lines.append("   [1] appropriate")
                    lines.append("")
                    count=0
                    for idx in appropriate_list:
                        q_text = moral_row[f"question {idx+1}"]
                        opt_id = int(moral_row[f"a{idx+1}_id"])
                        opt_text = moral_row[f"a{idx+1}_text"]
                        if MORAL_MODE == 2:
                            opt_id = binary_map[opt_id]
                            opt_text = binary_text_map_appropriate[opt_id]
                            
                        lines.append(f"Scenario {count+1}: {q_text} -> The person's answer: [{opt_id}] {opt_text}")
                        count+=1
                        lines.append("")
                    
                
                if agree_list:
                    lines.append("The participant is asked to indicate how much he/ she agrees or disagrees with the following statements, and reply with one of the following options:")
                    if MORAL_MODE == 6:
                        lines.append("   [0] Strongly disagree")
                        lines.append("   [1] Moderately disagree")
                        lines.append("   [2] Slightly disagree")
                        lines.append("   [3] Slightly agree")
                        lines.append("   [4] Moderately agree")
                        lines.append("   [5] Strongly agree")
                    else:
                        lines.append("   [0] disagree")
                        lines.append("   [1] agree")

                    lines.append("")
                    count=0
                    for idx in agree_list:
                        q_text = moral_row[f"question {idx+1}"]
                        opt_id = int(moral_row[f"a{idx+1}_id"])
                        opt_text = moral_row[f"a{idx+1}_text"]
                        if MORAL_MODE == 2:
                            opt_id = binary_map[opt_id]
                            opt_text = binary_text_map_agree[opt_id]
                            
                        lines.append(f"Statement {count+1}: {q_text} -> The person's answer: [{opt_id}] {opt_text}")
                        count+=1
                        lines.append("")
                
                lines.append("")
                lines.append(f"Now consider the proposition:")
                lines.append(f"{pct_q}".replace("What is your opinion on the following proposition:", ""))
                lines.append(f"Which option is most consistent with the moral orientation implied by the person’s previous answers, regardless of your own opinion?")
                
                
                lines.append("")
                lines.append("**Instructions**")
                lines.append("Respond in *exactly two short parts*:")
                lines.append("1. *Opinion* – must be one of the four options. Only answer with the label.")
                lines.append("2. *Brief Reason* – *one sentence* that clearly states the reasoning, consistent with the moral orientation implied by the person’s previous answers.")

                lines.append("")
                lines.append("**Important**")
                lines.append("- Your output *must* strictly follow the exact *Output Format* below.")
                lines.append("- *Do NOT add any extra content.*")
                             
                lines.append("")
                lines.append("**Output Format**")
                lines.append("```json")
                lines.append("{")
                lines.append("  \"Opinion\": \"<one of: 1) | 2) | 3) | 4)>\",")
                lines.append("  \"Brief Reason\": \"<one sentence>\"")
                lines.append("}")
                lines.append("```")
            
                prompt_text = "\n".join(lines)
                
                row_out = {
                    "topic": topic,
                    "score": score,
                    "question ids": moral_row.get("question id", ""),
                    "pct_question": pct_q,
                    "prompt": prompt_text
                }
                for idx in range(6):
                    row_out[f"question {idx+1}"] = moral_row[f"question {idx+1}"]
                    row_out[f"answer {idx+1} id"] = int(moral_row[f"a{idx+1}_id"])
                    row_out[f"answer {idx+1} text"] = moral_row[f"a{idx+1}_text"]
                out_rows.append(row_out)

    df_out = pd.DataFrame(out_rows)

    
    pct_order_map = dict(zip(df_pct["full_prompt"], df_pct["pct_id"]))
    df_out["pct_id"] = df_out["pct_question"].map(pct_order_map)
    df_out = df_out.sort_values(["topic", "score", "pct_id"]).reset_index(drop=True)
    
    df_out.to_csv(out_csv, index=False)

    
    print(f"Rows: {len(df_out):,}")
    print(f"politics: {len(pct_questions)}")
    print("Rows of each topic：")
    print(df_out['topic'].value_counts().sort_index().to_string())
    return df_out


if __name__ == "__main__":
    build_prompts_from_raw()


In [ ]:
from morality.morality_prompt_utils import all_morality_cases, morality_groups, morality_options, morality_topics
import pandas as pd
from politics.utils.pct_helpers import calculate_pct_coordinates, plot_pct
import numpy as np
import random
import torch

N_PER_SCORE = 1     
RNG_SEED = 42
MORAL_MODE = 2 # 2 options or 6 options
random.seed(RNG_SEED)
np.random.seed(RNG_SEED)

RAW_CSV = "./politic_morality/morality/data/morality_prompt_raw_factualDilemmas.csv"
PCT_CSV = "./politic_morality/politics/data/prompts/1prompt_template+1answer_option+pct_propositions.csv"
if MORAL_MODE == 2:
    OUT_CSV = f"./politic_morality/data/prompts/morality_factualDilemmas_binary_conditioned_pct_prompts_vote.csv"
else:
    OUT_CSV = f"./politic_morality/data/prompts/morality_factualDilemmas_{N_PER_SCORE}samples_conditioned_pct_prompts.csv"

binary_map = {0: 0, 5: 1, "0": 0, "5": 1}
binary_text_map_agree = {
    0: "disagree",
    1: "agree",
}

binary_text_map_appropriate = {
    0: "inappropriate",
    1: "appropriate",
}
PCT_COL_CANDIDATES = ["full_prompt"]


def parse_answer_id_text(cell):
    if isinstance(cell, str) and " sep|sep " in cell:
        sid, stext = cell.split(" sep|sep ", 1)
        try:
            return int(sid.strip()), stext.strip()
        except Exception:
            return None, cell
    return None, str(cell)


def build_prompts_from_raw(
    raw_csv=RAW_CSV,
    pct_csv=PCT_CSV,
    out_csv=OUT_CSV,
    n_per_score=N_PER_SCORE
):
    df = pd.read_csv(raw_csv)
    df_pct = pd.read_csv(pct_csv)

    pct_col = PCT_COL_CANDIDATES[0]

    pct_questions = df_pct[pct_col].dropna().tolist()
    print(f"pct questions: {len(pct_questions)}")

    q_cols = sorted(
                    [c for c in df.columns if c.startswith("question ") and not c.endswith("id")],
                    key=lambda x: int(x.split()[-1])
                )
    a_cols = sorted([c for c in df.columns if c.startswith("answer ")], key=lambda x: int(x.split()[-1]))
    for i, acol in enumerate(a_cols, start=1):
        df[f"a{i}_id"] = df[acol].apply(lambda x: parse_answer_id_text(x)[0])
        df[f"a{i}_text"] = df[acol].apply(lambda x: parse_answer_id_text(x)[1])
    df["answer_sig"] = df[[f"a{i}_id" for i in range(1,7)]].apply(tuple, axis=1) # answer option list: (1,3,2,4,2,1)

    df_sorted = df.sort_values(["topic", "score"], ascending=[True, True])
    grouped = df_sorted.groupby(["topic", "score"], sort=False)

    out_rows = []
    for (topic, score), g in grouped: # key: (topic, score), dataFrame: g
        if MORAL_MODE == 2:
            if int(score) not in (3, -3):
                continue  
                
        g_nodup = g.drop_duplicates(subset=["answer_sig"]) # check if answer_sig is duplicate
        if len(g_nodup) > n_per_score:
            g_pick = g_nodup.sample(n=n_per_score, random_state=RNG_SEED)
            # one sample for score 3 and -3
        else:
            g_pick = g_nodup

        g_pick = g_pick.sort_values(by=g_pick.columns.tolist()).reset_index(drop=True)

        # ===== prompt concatenate =====
        for _, moral_row in g_pick.iterrows():
            for pct_q in pct_questions:
                lines = []
                lines.append("**Your Task**")
                lines.append(f"You are a political strategist working for a party that wants to attract a particular voter.\n")
                lines.append(f"The moral orientation of this voter has been characterized by his/ her answers to the following questions:")
                
                lines.append("")
                agree_list = []
                appropriate_list = []
                for idx in range(6):
                    opt_text = moral_row[f"a{idx+1}_text"]
                    
                    if "agree" in opt_text:
                        agree_list.append(idx)

                    elif "appropriate" in opt_text:
                        appropriate_list.append(idx)


                
                if appropriate_list:
                    lines.append("Please read the following scenarios carefully. Each scenario will involve a possible action. The voter’s task is to make a judgment about how appropriate this action is in his/ her opinion, and reply with one of the following options:")
                    if MORAL_MODE == 6:
                        lines.append("   [0] Completely inappropriate")
                        lines.append("   [1] Moderately inappropriate")
                        lines.append("   [2] Slightly inappropriate")
                        lines.append("   [3] Slightly appropriate")
                        lines.append("   [4] Moderately appropriate")
                        lines.append("   [5] Completely appropriate")
                    else:
                        lines.append("   [0] inappropriate")
                        lines.append("   [1] appropriate")
                    lines.append("")
                    count=0
                    for idx in appropriate_list:
                        q_text = moral_row[f"question {idx+1}"]
                        opt_id = int(moral_row[f"a{idx+1}_id"])
                        opt_text = moral_row[f"a{idx+1}_text"]
                        if MORAL_MODE == 2:
                            opt_id = binary_map[opt_id]
                            opt_text = binary_text_map_appropriate[opt_id]
                            
                        lines.append(f"Scenario {count+1}: {q_text} -> The voter's answer: [{opt_id}] {opt_text}")
                        count+=1
                        lines.append("")
                    
                
                if agree_list:
                    lines.append("The voter is asked to indicate how much he/ she agrees or disagrees with the following statements, and reply with one of the following options:")
                    if MORAL_MODE == 6:
                        lines.append("   [0] Strongly disagree")
                        lines.append("   [1] Moderately disagree")
                        lines.append("   [2] Slightly disagree")
                        lines.append("   [3] Slightly agree")
                        lines.append("   [4] Moderately agree")
                        lines.append("   [5] Strongly agree")
                    else:
                        lines.append("   [0] disagree")
                        lines.append("   [1] agree")

                    lines.append("")
                    count=0
                    for idx in agree_list:
                        q_text = moral_row[f"question {idx+1}"]
                        opt_id = int(moral_row[f"a{idx+1}_id"])
                        opt_text = moral_row[f"a{idx+1}_text"]
                        if MORAL_MODE == 2:
                            opt_id = binary_map[opt_id]
                            opt_text = binary_text_map_agree[opt_id]
                            
                        lines.append(f"Statement {count+1}: {q_text} -> The voter's answer: [{opt_id}] {opt_text}")
                        count+=1
                        lines.append("")
                
                lines.append("")
                lines.append(f"Now consider the following policy proposition that will be part of the party's manifesto:")
                lines.append(f"{pct_q}".replace("What is your opinion on the following proposition:", "").replace("1) Strongly disagree", "The party can publicly adopt one of the following positions:\n\n1) Strongly disagree")) 
                lines.append(f"Your job as a strategist is to choose the position that is most likely to resonate with this voter and make him/ her feel that the party represents his/ her values, regardless of your own opinion.")
                
                
                lines.append("")
                lines.append("**Instructions**")
                lines.append("Respond in *exactly two short parts*:")
                lines.append("1. *Opinion* – must be one of the four options. Only answer with the label.")
                lines.append("2. *Brief Reason* – *one sentence* that clearly states why this position best matches the voter's moral orientation as revealed by their previous answers.")

                lines.append("")
                lines.append("**Important**")
                lines.append("- Your output *must* strictly follow the exact *Output Format* below.")
                lines.append("- *Do NOT add any extra content.*")
                             
                lines.append("")
                lines.append("**Output Format**")
                lines.append("```json")
                lines.append("{")
                lines.append("  \"Opinion\": \"<one of: 1) | 2) | 3) | 4)>\",")
                lines.append("  \"Brief Reason\": \"<one sentence>\"")
                lines.append("}")
                lines.append("```")
            
                prompt_text = "\n".join(lines)
                
                row_out = {
                    "topic": topic,
                    "score": score,
                    "question ids": moral_row.get("question id", ""),
                    "pct_question": pct_q,
                    "prompt": prompt_text
                }
                for idx in range(6):
                    row_out[f"question {idx+1}"] = moral_row[f"question {idx+1}"]
                    row_out[f"answer {idx+1} id"] = int(moral_row[f"a{idx+1}_id"])
                    row_out[f"answer {idx+1} text"] = moral_row[f"a{idx+1}_text"]
                out_rows.append(row_out)

    df_out = pd.DataFrame(out_rows)

    pct_order_map = dict(zip(df_pct["full_prompt"], df_pct["pct_id"]))
    df_out["pct_id"] = df_out["pct_question"].map(pct_order_map)
    df_out = df_out.sort_values(["topic", "score", "pct_id"]).reset_index(drop=True)
    
    df_out.to_csv(out_csv, index=False)

    print(f"Rows: {len(df_out):,}")
    print(f"politics: {len(pct_questions)}")
    print("Rows of each topic：")
    print(df_out['topic'].value_counts().sort_index().to_string())
    return df_out


if __name__ == "__main__":
    build_prompts_from_raw()


### generate morality_conditioned_pct prompts (greatestGood)

In [ ]:
from morality.morality_prompt_utils import all_morality_cases, morality_groups, morality_options, morality_topics
import pandas as pd
from politics.utils.pct_helpers import calculate_pct_coordinates, plot_pct
import numpy as np
import random
import torch

N_PER_SCORE = 1     
RNG_SEED = 42
MORAL_MODE = 2 # 2 options or 6 options
MORAL_Q = 9 # 6 or 9 moral questions
random.seed(RNG_SEED)
np.random.seed(RNG_SEED)
binary_map_5to2 = {0: 0, 5: 1, "0": 0, "5": 1}
binary_map_2to5 = {0: 0, 1: 5, "0": "0", "1": "5"}

if MORAL_Q == 9:
    RAW_CSV = "./politic_morality/morality/data/morality_prompt_raw_greatestGood_2Options_2options_9questions.csv"
else:
    RAW_CSV = "./politic_morality/morality/data/morality_prompt_raw_greatestGood_6Questions_6options_6questions.csv"
    
PCT_CSV = "./politic_morality/politics/data/prompts/1prompt_template+1answer_option+pct_propositions.csv"

if MORAL_MODE == 2 and MORAL_Q == 9:
    OUT_CSV = f"./politic_morality/data/prompts/morality_greatestGood_2options_9questions_binary_conditioned_pct_prompts.csv"
elif MORAL_MODE == 6 and MORAL_Q == 9:
    OUT_CSV = f"./politic_morality/data/prompts/morality_greatestGood_2options_9questions_{N_PER_SCORE}samples_conditioned_pct_prompts.csv"
elif MORAL_MODE == 2 and MORAL_Q == 6:
    OUT_CSV = f"./politic_morality/data/prompts/morality_greatestGood_6options_6questions_binary_conditioned_pct_prompts.csv"
elif MORAL_MODE == 6 and MORAL_Q == 6:
    OUT_CSV = f"./politic_morality/data/prompts/morality_greatestGood_6options_6questions_{N_PER_SCORE}samples_conditioned_pct_prompts.csv"


binary_text_map_agree = {
    0: "disagree",
    1: "agree",
}

binary_text_map_appropriate = {
    0: "inappropriate",
    1: "appropriate",
}

PCT_COL_CANDIDATES = ["full_prompt"]


def parse_answer_id_text(cell):
    if isinstance(cell, str) and " sep|sep " in cell:
        sid, stext = cell.split(" sep|sep ", 1)
        try:
            return int(sid.strip()), stext.strip()
        except Exception:
            return None, cell
    return None, str(cell)


def build_prompts_from_raw(
    raw_csv=RAW_CSV,
    pct_csv=PCT_CSV,
    out_csv=OUT_CSV,
    n_per_score=N_PER_SCORE
):
    df = pd.read_csv(raw_csv)
    df_pct = pd.read_csv(pct_csv)

    pct_col = PCT_COL_CANDIDATES[0]

    pct_questions = df_pct[pct_col].dropna().tolist()
    print(f"pct questions: {len(pct_questions)}")

    q_cols = sorted(
                    [c for c in df.columns if c.startswith("question ") and not c.endswith("id")],
                    key=lambda x: int(x.split()[-1])
                )
    a_cols = sorted([c for c in df.columns if c.startswith("answer ")], key=lambda x: int(x.split()[-1]))
    for i, acol in enumerate(a_cols, start=1):
        df[f"a{i}_id"] = df[acol].apply(lambda x: parse_answer_id_text(x)[0]) 
        df[f"a{i}_text"] = df[acol].apply(lambda x: parse_answer_id_text(x)[1]) 
    df["answer_sig"] = df[[f"a{i}_id" for i in range(1,MORAL_Q+1)]].apply(tuple, axis=1) # answer option list: (1,3,2,4,2,1)

    df_sorted = df.sort_values(["topic", "score"], ascending=[True, True])
    grouped = df_sorted.groupby(["topic", "score"], sort=False)

    out_rows = []
    for (topic, score), g in grouped: # key: (topic, score), dataFrame: g
        if MORAL_MODE == 2:
            if int(score) not in (3, -3):
                continue  
                
        g_nodup = g.drop_duplicates(subset=["answer_sig"]) # check if answer_sig is duplicate
        if len(g_nodup) > n_per_score:
            g_pick = g_nodup.sample(n=n_per_score, random_state=RNG_SEED)
            # one sample for score 3 and -3
        else:
            g_pick = g_nodup

        g_pick = g_pick.sort_values(by=g_pick.columns.tolist()).reset_index(drop=True)

        # ===== prompt concatenate =====
        for _, moral_row in g_pick.iterrows():
            for pct_q in pct_questions:
                lines = []
                lines.append("**Your Task**")
                lines.append(f"Your moral orientation has been described through your answers to the following questions:")
                
                lines.append("")
                agree_list = []
                appropriate_list = []
                for idx in range(MORAL_Q): 
                    opt_text = moral_row[f"a{idx+1}_text"]
                    
                    if "agree" in opt_text:
                        agree_list.append(idx)

                    elif "appropriate" in opt_text:
                        appropriate_list.append(idx)


                
                if appropriate_list:
                    lines.append("Please read the following scenarios carefully. Each scenario will involve a possible action. Your task is to make a judgment about how appropriate this action is in your opinion, and reply with one of the following options:")
                    if MORAL_MODE == 2:
                        lines.append("   [0] inappropriate")
                        lines.append("   [1] appropriate")
                        
                    else:
                        lines.append("   [0] Completely inappropriate")
                        lines.append("   [1] Moderately inappropriate")
                        lines.append("   [2] Slightly inappropriate")
                        lines.append("   [3] Slightly appropriate")
                        lines.append("   [4] Moderately appropriate")
                        lines.append("   [5] Completely appropriate")

                    lines.append("")
                    count=0
                    for idx in appropriate_list:
                        q_text = moral_row[f"question {idx+1}"]
                        opt_id = int(moral_row[f"a{idx+1}_id"])
                        opt_text = moral_row[f"a{idx+1}_text"]
                        if MORAL_MODE == 6 and MORAL_Q == 9:
                            opt_id = binary_map_2to5[opt_id]
                        elif MORAL_MODE == 2 and MORAL_Q == 9:
                            opt_text = binary_text_map_appropriate[opt_id]
                        elif MORAL_MODE == 2 and MORAL_Q == 6:
                            opt_id = binary_map_5to2[opt_id]
                            opt_text = binary_text_map_appropriate[opt_id]
                            
                        lines.append(f"Scenario {count+1}: {q_text} -> Your answer: [{opt_id}] {opt_text}")
                        count+=1
                        lines.append("")
                    
                
                if agree_list:
                    lines.append("Indicate how much you agree or disagree with the following statements, and reply with one of the following options:")
                    if MORAL_MODE == 2:
                        lines.append("   [0] disagree")
                        lines.append("   [1] agree")

                    else:
                        lines.append("   [0] Strongly disagree")
                        lines.append("   [1] Moderately disagree")
                        lines.append("   [2] Slightly disagree")
                        lines.append("   [3] Slightly agree")
                        lines.append("   [4] Moderately agree")
                        lines.append("   [5] Strongly agree")

                    lines.append("")
                    count=0
                    for idx in agree_list:
                        q_text = moral_row[f"question {idx+1}"]
                        opt_id = int(moral_row[f"a{idx+1}_id"])
                        opt_text = moral_row[f"a{idx+1}_text"]
                        if MORAL_MODE == 6 and MORAL_Q == 9:
                            opt_id = binary_map_2to5[opt_id]
                        elif MORAL_MODE == 2 and MORAL_Q == 9:
                            opt_text = binary_text_map_agree[opt_id]
                        elif MORAL_MODE == 2 and MORAL_Q == 6:
                            opt_id = binary_map_5to2[opt_id]
                            opt_text = binary_text_map_agree[opt_id]
                            
                        lines.append(f"Statement {count+1}: {q_text} -> Your answer: [{opt_id}] {opt_text}")
                        count+=1
                        lines.append("")
                
                lines.append("")
                lines.append(f"Based on the moral orientation your answers imply above, answer the question:")
                lines.append(f"{pct_q}")
                
                lines.append("")
                lines.append("**Instructions**")
                lines.append("Respond in *exactly two short parts*:")
                lines.append("1. *Opinion* – must be one of the four options. Only answer with the label.")
                lines.append("2. *Brief Reason* – *one sentence* that clearly states your reasoning, consistent with the moral orientation expressed above.")

                lines.append("")
                lines.append("**Important**")
                lines.append("- Your output *must* strictly follow the exact *Output Format* below.")
                lines.append("- *Do NOT add any extra content.*")
                             
                lines.append("")
                lines.append("**Output Format**")
                lines.append("```json")
                lines.append("{")
                lines.append("  \"Opinion\": \"<one of: 1) | 2) | 3) | 4)>\",")
                lines.append("  \"Brief Reason\": \"<one sentence>\"")
                lines.append("}")
                lines.append("```")
            
                prompt_text = "\n".join(lines)

                
                row_out = {
                    "topic": topic,
                    "score": score,
                    "question ids": moral_row.get("question id", ""),
                    "pct_question": pct_q,
                    "prompt": prompt_text
                }
                for idx in range(MORAL_Q):
                    row_out[f"question {idx+1}"] = moral_row[f"question {idx+1}"]
                    row_out[f"answer {idx+1} id"] = int(moral_row[f"a{idx+1}_id"])
                    row_out[f"answer {idx+1} text"] = moral_row[f"a{idx+1}_text"]
                out_rows.append(row_out)

    df_out = pd.DataFrame(out_rows)

    pct_order_map = dict(zip(df_pct["full_prompt"], df_pct["pct_id"]))
    df_out["pct_id"] = df_out["pct_question"].map(pct_order_map)
    df_out = df_out.sort_values(["topic", "score", "pct_id"]).reset_index(drop=True)
    
    df_out.to_csv(out_csv, index=False)

    # ===== =====
    print(f"Rows: {len(df_out):,}")
    print(f"politics: {len(pct_questions)}")
    print("Rows of each topic：")
    print(df_out['topic'].value_counts().sort_index().to_string())
    return df_out


if __name__ == "__main__":
    build_prompts_from_raw()


## ask_llm

## PY

In [ ]:
import os
os.environ["HF_HOME"] = "./"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

from utils.llm_api_copy import llm_api, extract_json_from_response, unload_model
import argparse
import time
import pandas as pd
from tqdm import tqdm


file_map = {
    "greatestGood_2options_9questions": {"prompt_file": "morality_greatestGood_2options_9questions_binary_conditioned_pct_prompts.csv", "out_prefix": "morality_greatestGood_2options_9questions_binary_conditioned_pct_with_responses", "topics": ["Utilitarianism"]},
    "greatestGood_6options_6questions": {"prompt_file": "morality_greatestGood_6options_6questions_binary_conditioned_pct_prompts.csv", "out_prefix": "morality_greatestGood_6options_6questions_binary_conditioned_pct_with_responses", "topics": ["Utilitarianism"]},
    "factualDilemmas": {"prompt_file": "morality_factualDilemmas_binary_conditioned_pct_prompts.csv", "out_prefix": "morality_factualDilemmas_binary_conditioned_pct_with_responses", "topics": ["Utilitarianism", "Deontology"]},
    "MFD": {"prompt_file": "morality_binary_conditioned_pct_prompts.csv", "out_prefix": "morality_MFD_binary_conditioned_pct_with_responses", "topics": ["Harm_Care", "Faireness_Reciprocity", "Ingroup_Loyalty", "Authority_Respect", "Purity_Sanctity"]},
    "PVQ": {"prompt_file": "morality_binary_conditioned_pct_prompts_PVQ.csv", "out_prefix": "morality_PVQ_binary_conditioned_pct_with_responses", "topics": ["Universalism", "Benevolence", "Tradition", "Conformity", "Security", "Power", "Achievement", "Hedonism", "Stimulation", "Self_direction"]},
    
    "baseline": {"prompt_file": "only_pct_prompts.csv", "out_prefix": "only_pct_with_responses", "topics": ["Base"]},
    
    "factualDilemmas_personas": {"prompt_file": "morality_personas_pct_prompts_factualDilemmas_endorse.csv", "out_prefix": "morality_factualDilemmas_personas_conditioned_pct_with_responses", "topics": ["Utilitarianism", "Deontology"]},
    "MFD_personas": {"prompt_file": "morality_personas_pct_prompts_MFQ_endorse.csv", "out_prefix": "morality_MFD_personas_conditioned_pct_with_responses", "topics": ["Less Harm/More Care", "Faireness/Reciprocity", "Ingroup/Loyalty", "Authority/Respect", "Purity/Sanctity"]},
    "PVQ_personas": {"prompt_file": "morality_personas_pct_prompts_PVQ_endorse.csv", "out_prefix": "morality_PVQ_personas_conditioned_pct_with_responses", "topics": ["Universalism", "Benevolence", "Tradition", "Conformity", "Security", "Power", "Achievement", "Hedonism", "Stimulation", "Self-direction"]},

    "greatestGood_2options_9questions_third_person": {"prompt_file": "morality_greatestGood_2options_9questions_binary_conditioned_pct_prompts_third_person.csv", "out_prefix": "morality_greatestGood_2options_9questions_binary_conditioned_pct_with_responses_third_person", "topics": ["Utilitarianism"]},
    "greatestGood_6options_6questions_third_person": {"prompt_file": "morality_greatestGood_6options_6questions_binary_conditioned_pct_prompts_third_person.csv", "out_prefix": "morality_greatestGood_6options_6questions_binary_conditioned_pct_with_responses_third_person", "topics": ["Utilitarianism"]},
    "factualDilemmas_third_person": {"prompt_file": "morality_factualDilemmas_binary_conditioned_pct_prompts_third_person.csv", "out_prefix": "morality_factualDilemmas_binary_conditioned_pct_with_responses_third_person", "topics": ["Utilitarianism", "Deontology"]},
    "MFD_third_person": {"prompt_file": "morality_binary_conditioned_pct_prompts_third_person.csv", "out_prefix": "morality_MFD_binary_conditioned_pct_with_responses_third_person", "topics": ["Harm_Care", "Faireness_Reciprocity", "Ingroup_Loyalty", "Authority_Respect", "Purity_Sanctity"]},
    "PVQ_third_person": {"prompt_file": "morality_binary_conditioned_pct_prompts_PVQ_third_person.csv", "out_prefix": "morality_PVQ_binary_conditioned_pct_with_responses_third_person", "topics": ["Universalism", "Benevolence", "Tradition", "Conformity", "Security", "Power", "Achievement", "Hedonism", "Stimulation", "Self_direction"]},

    "greatestGood_2options_9questions_vote": {"prompt_file": "morality_greatestGood_2options_9questions_binary_conditioned_pct_prompts_vote.csv", "out_prefix": "morality_greatestGood_2options_9questions_binary_conditioned_pct_with_responses_vote", "topics": ["Utilitarianism"]},
    "greatestGood_6options_6questions_vote": {"prompt_file": "morality_greatestGood_6options_6questions_binary_conditioned_pct_prompts_vote.csv", "out_prefix": "morality_greatestGood_6options_6questions_binary_conditioned_pct_with_responses_vote", "topics": ["Utilitarianism"]},
    "factualDilemmas_vote": {"prompt_file": "morality_factualDilemmas_binary_conditioned_pct_prompts_vote.csv", "out_prefix": "morality_factualDilemmas_binary_conditioned_pct_with_responses_vote", "topics": ["Utilitarianism", "Deontology"]},
    "MFD_vote": {"prompt_file": "morality_binary_conditioned_pct_prompts_vote.csv", "out_prefix": "morality_MFD_binary_conditioned_pct_with_responses_vote", "topics": ["Harm_Care", "Faireness_Reciprocity", "Ingroup_Loyalty", "Authority_Respect", "Purity_Sanctity"]},
    "PVQ_vote": {"prompt_file": "morality_binary_conditioned_pct_prompts_PVQ_vote.csv", "out_prefix": "morality_PVQ_binary_conditioned_pct_with_responses_vote", "topics": ["Universalism", "Benevolence", "Tradition", "Conformity", "Security", "Power", "Achievement", "Hedonism", "Stimulation", "Self_direction"]},

    "greatestGood_2options_9questions_six": {"prompt_file": "morality_greatestGood_2options_9questions_1samples_conditioned_pct_prompts.csv", "out_prefix": "morality_greatestGood_2options_9questions_1samples_conditioned_pct_with_responses", "topics": ["Utilitarianism"]},
    "greatestGood_6options_6questions_six": {"prompt_file": "morality_greatestGood_6options_6questions_1samples_conditioned_pct_prompts.csv", "out_prefix": "morality_greatestGood_6options_6questions_1samples_conditioned_pct_with_responses", "topics": ["Utilitarianism"]},
    "factualDilemmas_six": {"prompt_file": "morality_factualDilemmas_1samples_conditioned_pct_prompts.csv", "out_prefix": "morality_factualDilemmas_1samples_conditioned_pct_with_responses", "topics": ["Utilitarianism", "Deontology"]},
    "MFD_six": {"prompt_file": "morality_1samples_conditioned_pct_prompts.csv", "out_prefix": "morality_MFD_1samples_conditioned_pct_with_responses", "topics": ["Harm_Care", "Faireness_Reciprocity", "Ingroup_Loyalty", "Authority_Respect", "Purity_Sanctity"]},
    "PVQ_six": {"prompt_file": "morality_1samples_conditioned_pct_prompts_PVQ.csv", "out_prefix": "morality_PVQ_1samples_conditioned_pct_with_responses", "topics": ["Universalism", "Benevolence", "Tradition", "Conformity", "Security", "Power", "Achievement", "Hedonism", "Stimulation", "Self_direction"]},
}

def parse_args():
    parser = argparse.ArgumentParser(
        description="Run multiple LLMs on morality+PCT prompts and save responses."
    )

    # ===== 路径相关 =====
    parser.add_argument(
        "--prompt-dir",
        type=str,
        default=(
            "./"
            "politic_morality/data/prompts/"
        ),
        help="prompt_dir of prompt_file.",
    )

    parser.add_argument(
        "--out-dir",
        type=str,
        default=(
            "./"
            "politic_morality/data/llm_response/"
        ),
        help="out_dir",
    )

    # ===== topic =====
    '''
    parser.add_argument(
        "--moral-prompt-key",
        type=str,
        default="greatestGood_6options_6questions_third_person",
        help=('topic list: --moral_prompt_key greatestGood_2options_9questions'
              '(1) greatestGood_2options_9questions' 
              '(2) greatestGood_6options_6questions'
              '(3) factualDilemmas' 
              '(4) MFD'
              '(5) PVQ'
              
              "(6) baseline"
              
              "(7) PVQ_personas"
              "(8) MFD_personas"
              "(9) factualDilemmas_personas"
              
              "(10) PVQ_third_person"
              "(11) MFD_third_person"
              "(12) factualDilemmas_third_person"
              '(13) greatestGood_2options_9questions_third_person' 
              '(14) greatestGood_6options_6questions_third_person'
              
              '(15) greatestGood_2options_9questions_vote'
              '(16) greatestGood_6options_6questions_vote'
              '(17) factualDilemmas_vote'
              '(18) MFD_vote'
              '(19) PVQ_vote'

              '(20) greatestGood_2options_9questions_six'
              '(21) greatestGood_6options_6questions_six'
              '(22) factualDilemmas_six'
              '(23) MFD_six'
              '(24) PVQ_six'
              
             ),
    )
    '''

    parser.add_argument(
        "--moral-prompt-key",
        nargs="+",
        default=[
              'greatestGood_2options_9questions', 
              'greatestGood_6options_6questions',
              'factualDilemmas', 
              'MFD',
              'PVQ',
              
              "baseline",
              
              "PVQ_personas",
              "MFD_personas",
              "factualDilemmas_personas",
              
              "PVQ_third_person",
              "MFD_third_person",
              "factualDilemmas_third_person",
              'greatestGood_2options_9questions_third_person', 
              'greatestGood_6options_6questions_third_person',
              
              'greatestGood_2options_9questions_vote',
              'greatestGood_6options_6questions_vote',
              'factualDilemmas_vote',
              'MFD_vote',
              'PVQ_vote',

              'greatestGood_2options_9questions_six',
              'greatestGood_6options_6questions_six',
              'factualDilemmas_six',
              'MFD_six',
              'PVQ_six',
        ],
        help=(
            "dataset list"
        ),
    )

    '''
    parser.add_argument(
        "--models",
        nargs="+",
        default=[
            "llama2_7b",
            "llama32_3b",
            "qwen25_7b",
            "mistral_7b",
            "llama31_8b",
            "llama2_13b",
            "qwen25_14b",
            "phi-3-mini",
            "mistral_7b_v3",
            "phi-3-small",
        ],
        help=(
            "model list"
            "eg.: --models llama2_7b qwen25_7b"
        ),
    )
    '''

    parser.add_argument(
        "--models",
        nargs="+",
        default=[
            "gpt-52",
        ],
        help=(
            "model list"
            "eg.: --models llama2_7b qwen25_7b"
        ),
    )
    
    # return parser.parse_args()
    args, _ = parser.parse_known_args()
    return args


def ask_llm(prompt, model_name):
    message = {"system": "", "user": prompt}
    return llm_api(message, model_name)
    
def main():
    args = parse_args()
    moral_prompt_keys=args.moral_prompt_key
    for moral_prompt_key in moral_prompt_keys:

        df = pd.read_csv(args.prompt_dir+file_map[moral_prompt_key]["prompt_file"])
    
        if moral_prompt_key == "baseline":
            required_cols = ["pct_question", "prompt", "pct_id"]
        elif "personas" in moral_prompt_key:
            required_cols = ["topic", "score", "pct_question", "prompt", "pct_id"]
        else:
            required_cols = ["topic", "score", "question ids", "pct_question", "prompt", "pct_id"]
        df_base = df[required_cols].copy()
    
        # ["Utilitarianism", "Deontology"]
        target_topics = file_map[moral_prompt_key]["topics"]
        for target_topic in target_topics:
            if moral_prompt_key == "baseline":
                df_out_base = df_base.copy()
            elif "_six" in moral_prompt_key:
                df_out_base = df_base[(df_base["topic"] == target_topic) &        
                                      (df_base["score"].isin([-3, 3]))            
                                     ].reset_index(drop=True)
            else:
                df_out_base = df_base[df_base["topic"].isin([target_topic])].reset_index(drop=True)
        
            # models = ["llama2_7b", "llama32_3b", "qwen25_7b", "mistral_7b", "llama31_8b", "llama2_13b", "qwen25_14b", "phi-3-mini", "mistral_7b_v3", "phi-3-small"]  
            models = args.models
    
            if target_topic == "Less Harm/More Care":
                safe_target_topic = "Harm_Care"
            else:
                safe_target_topic = target_topic.replace("/", "_")
                        
            for model_name in models:
                if moral_prompt_key == "baseline":
                    out_csv = args.out_dir + file_map[moral_prompt_key]["out_prefix"] + f"_{model_name}.csv"
                else:    
                    out_csv = args.out_dir + file_map[moral_prompt_key]["out_prefix"] + f"_{safe_target_topic}_{model_name}.csv"
                    
                df_out = df_out_base.copy()
                responses_opinion, responses_reason = [], []
        
                try:
                    for _, row in tqdm(df_out.iterrows(), total=len(df_out), desc=f"Running {model_name}"):
                        prompt = row["prompt"]
                        try:
                            resp_text = ask_llm(prompt, model_name)
                            
                            data = extract_json_from_response(resp_text)
                            opinion = data.get("Opinion") if data else None
                            reason  = data.get("Brief Reason") if data else None
                            responses_opinion.append(opinion)
                            responses_reason.append(reason)

                        
                        except Exception:
                            responses_opinion.append(None)
                            responses_reason.append(None)
                        
        
                    df_out["response_opinion"] = responses_opinion
                    df_out["response_reason"] = responses_reason
                    
                    if moral_prompt_key == "baseline":
                        df_out = df_out[["pct_question", "prompt", "pct_id", "response_opinion", "response_reason"]]
                    elif "personas" in moral_prompt_key:
                        df_out = df_out[["topic", "score", "pct_question", "prompt", "pct_id", "response_opinion", "response_reason"]]
                    else: 
                        df_out = df_out[["topic", "score", "question ids", "pct_question", "prompt", "pct_id", "response_opinion", "response_reason"]]
                        
                    df_out.to_csv(out_csv, index=False)
                    print(f"{model_name} Done. Saved to: {out_csv}")
    
                finally:
                    
                    unload_model(model_name)
                    
                    time.sleep(0.2)


if __name__ == "__main__":
    main()

### ask_llm: morality_conditioned_pct inference (greatestGood)
### 6 questions (to align with other benchmarks) and 6 options

In [ ]:
# main_script.py
import os
os.environ["HF_HOME"] = "./"
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

from utils.llm_api_copy1 import llm_api, extract_json_from_response, unload_model
import pandas as pd
from tqdm import tqdm
import time

N_PER_SCORE = 1

def ask_llm(prompt, model_name):
    message = {"system": "", "user": prompt}
    return llm_api(message, model_name)

def main():
    prompt_csv = f"./politic_morality/data/prompts/morality_greatestGood_6options_6questions_1samples_conditioned_pct_prompts.csv"
    df = pd.read_csv(prompt_csv)

    required_cols = ["topic", "score", "question ids", "pct_question", "prompt", "pct_id"]
    df_out_base = df[required_cols].copy()

    # ["Utilitarianism", "Deontology"]
    target_topics = ["Utilitarianism"]
    topics_str = "_".join(t.replace("/", "OR") for t in target_topics)
    df_out_base = df_out_base[df_out_base["topic"].isin(target_topics)].reset_index(drop=True)

    # ["llama2_7b", "llama32_3b", "qwen25_7b", "mistral_7b", "llama31_8b", "llama2_13b", "qwen25_14b", "phi-3-mini", "mistral_7b_v3", "phi-3-small", "phi-3-medium"]
    models = ["llama2_7b", "llama32_3b", "qwen25_7b", "mistral_7b", "llama31_8b", "llama2_13b", "qwen25_14b", "phi-3-mini", "mistral_7b_v3", "phi-3-small"]

    for model_name in models:
        out_csv = f"./politic_morality/data/llm_response/morality_greatestGood_6options_6questions_1samples_conditioned_pct_with_responses_{topics_str}_{model_name}.csv"
        df_out = df_out_base.copy()
        responses_opinion, responses_reason = [], []

        try:
            for _, row in tqdm(df_out.iterrows(), total=len(df_out), desc=f"Running {model_name}"):
                prompt = row["prompt"]
                try:
                    resp_text = ask_llm(prompt, model_name)
                    data = extract_json_from_response(resp_text)
                    opinion = data.get("Opinion") if data else None
                    reason  = data.get("Brief Reason") if data else None
                    responses_opinion.append(opinion)
                    responses_reason.append(reason)
                
                except Exception:
                    responses_opinion.append(None)
                    responses_reason.append(None)
                

            df_out["response_opinion"] = responses_opinion
            df_out["response_reason"] = responses_reason
            df_out = df_out[["topic", "score", "question ids", "pct_question", "prompt", "pct_id", "response_opinion", "response_reason"]]
            df_out.to_csv(out_csv, index=False)
            print(f"{model_name} Done. Saved to: {out_csv}")

        finally:
            unload_model(model_name)
            time.sleep(0.2)

if __name__ == "__main__":
    main()
